# 1.1 Verosimilitud de un cliente

$$L(\lambda,p \mid x,t_x,T) = (1-p)^x \lambda^x e^{-\lambda T} + \delta_{x>0}\, p(1-p)^{x-1}\lambda^x e^{-\lambda t_x}$$

Hay dos historias que producen los mismos datos $(x, t_x, T)$:

**1. El cliente sigue vivo al final de la calibración (primer término).**
El factor $(1-p)^x$ es la probabilidad de que el cliente no haya muerto después de ninguna de sus $x$ compras repetidas. El factor $\lambda^x e^{-\lambda T}$ es la verosimilitud de Poisson de observar $x$ compras en $(0, T]$. Como $e^{-\lambda T} = e^{-\lambda t_x}\, e^{-\lambda (T - t_x)}$, incluye que el proceso corrió hasta $t_x$ y que no hubo compras entre $t_x$ y $T$.

**2. El cliente murió justo después de su última compra, en $t_x$ (segundo término).**
El cliente sobrevivió a las primeras $x-1$ compras repetidas, con probabilidad $(1-p)^{x-1}$, y murió justo después de la $x$-ésima, con probabilidad $p$. Como ya no compra después de morir, el proceso de Poisson solo cuenta hasta $t_x$, y por eso aparece $e^{-\lambda t_x}$ en lugar de $e^{-\lambda T}$.

**3. Por qué el segundo término no existe cuando $x = 0$.**
En el BG/NBD el cliente solo puede morir justo después de una compra **repetida**. Si $x = 0$, el cliente solo hizo su compra inicial y no hubo ningún momento en que pudiera morir, así que la única historia posible es que sigue vivo. Por eso $\delta_{x>0} = 0$ y solo queda el primer término. (Esta es la pieza que cambia el MBG/NBD en 4.6.)

# 1.2 La mezcla

Como $\lambda$ y $p$ son independientes (supuesto v), cada término se integra por separado: $\lambda$ con la densidad $\text{Gamma}(r,\alpha)$ y $p$ con la densidad $\text{Beta}(a,b)$.

**Primer término** (cliente vivo en $T$):
- En $\lambda$: $\lambda^{x}e^{-\lambda T}\cdot\frac{\alpha^r}{\Gamma(r)}\lambda^{r-1}e^{-\alpha\lambda} \Rightarrow \frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)\,(\alpha+T)^{r+x}}$
- En $p$: $(1-p)^x\cdot\frac{p^{a-1}(1-p)^{b-1}}{B(a,b)} \Rightarrow \frac{B(a,\,b+x)}{B(a,b)}$

**Segundo término** (murió en $t_x$, solo si $x>0$):
- En $\lambda$: igual, con $t_x$ en lugar de $T$: $\frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)\,(\alpha+t_x)^{r+x}}$
- En $p$: $p\,(1-p)^{x-1}\cdot p^{a-1}(1-p)^{b-1} = p^{a}(1-p)^{b+x-2} \Rightarrow \frac{B(a+1,\,b+x-1)}{B(a,b)}$

**Resultado:**
$$L(r,\alpha,a,b \mid x,t_x,T)=\frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)}\left[\frac{B(a,b+x)}{B(a,b)}\frac{1}{(\alpha+T)^{r+x}}+\delta_{x>0}\frac{B(a+1,b+x-1)}{B(a,b)}\frac{1}{(\alpha+t_x)^{r+x}}\right]$$

In [ ]:
import sys; sys.path.append("..")
from src.data_prep import cargar_transacciones, construir_resumen
from src.bgnbd import log_verosimilitud

df = cargar_transacciones("../data/raw/CDNOW_sample.txt")
res = construir_resumen(df)
x, t_x, T = res["x"].values, res["t_x"].values, res["T"].values

ll = log_verosimilitud([0.243, 4.414, 0.793, 2.426], x, t_x, T)
print(f"Log-verosimilitud: {ll:.2f}")

In [ ]:
import numpy as np
from scipy.optimize import minimize

def neg_ll(log_params):
    params = np.exp(log_params)                     # siempre positivos
    return -log_verosimilitud(params, x, t_x, T)    # minimizamos el negativo

inicios = [np.zeros(4), np.array([-1, 1, -0.5, 1]), np.ones(4)]

resultados = []
for inicio in inicios:
    res_opt = minimize(neg_ll, inicio, method="Nelder-Mead",
                       options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8})
    resultados.append(res_opt)
    print(np.exp(res_opt.x).round(4), -res_opt.fun, res_opt.success)

mejor = min(resultados, key=lambda o: o.fun)
r_hat, alpha_hat, a_hat, b_hat = np.exp(mejor.x)
ll_max = -mejor.fun
print(f"\nr={r_hat:.3f}  alpha={alpha_hat:.3f}  a={a_hat:.3f}  b={b_hat:.3f}  LL={ll_max:.1f}")

In [ ]:
import sys; sys.path.append("..")
import numpy as np
from scipy.optimize import minimize
from src.bgnbd import log_verosimilitud

def simular_cliente(r, alpha, a, b, T, rng):
    lam = rng.gamma(shape=r, scale=1/alpha)       # λ ~ Gamma(forma r, tasa α) -> scale = 1/α
    p   = rng.beta(a, b)

    t, x, t_x = 0.0, 0, 0.0
    while True:
        t += rng.exponential(scale=1/lam)       # tiempo entre compras ~ Exponencial(tasa λ) -> scale = 1/λ
        if t > T:
            break
        x += 1
        t_x = t
        if rng.random() < p:                 # muere tras esta compra con probabilidad p
            break
    return x, t_x

In [ ]:
def simular_poblacion(n, r, alpha, a, b, rng):
    T = rng.uniform(27, 39, size=n)
    xs, txs = np.empty(n), np.empty(n)
    for i in range(n):
        xs[i], txs[i] = simular_cliente(r, alpha, a, b, T[i], rng)
    return xs, txs, T

def estimar(x, t_x, T):
    mejor = None
    for inicio in ([0, 0, 0, 0], [-1, 1, -0.5, 1]):
        o = minimize(lambda lp: -log_verosimilitud(np.exp(lp), x, t_x, T), inicio,
                     method="Nelder-Mead", options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8})
        if mejor is None or o.fun < mejor.fun:
            mejor = o
    return np.exp(mejor.x)

verdaderos = (0.25, 4.0, 0.8, 2.5)         # r, alpha, a, b
estimados = []
for semilla in range(30):
    rng = np.random.default_rng(semilla)
    x_sim, t_x_sim, T_sim = simular_poblacion(2357, *verdaderos, rng)
    estimados.append(estimar(x_sim, t_x_sim, T_sim))

estimados = np.array(estimados)
print("Verdaderos:", verdaderos)
print("Media:     ", estimados.mean(axis=0).round(3))
print("Desv. est.:", estimados.std(axis=0, ddof=1).round(3))

# 1.3 Simulación: ¿recuperamos los parámetros?

**Diseño.** Se simularon 2,357 clientes con parámetros verdaderos $(r,\alpha,a,b)=(0.25,\,4.0,\,0.8,\,2.5)$ y $T$ uniforme entre 27 y 39 semanas. Para cada cliente se sorteó $\lambda\sim\text{Gamma}(r,\alpha)$ y $p\sim\text{Beta}(a,b)$, y se simuló su historia compra por compra: tiempos entre compras exponenciales de tasa $\lambda$ y, después de cada compra, muerte con probabilidad $p$. Los parámetros se estimaron con la verosimilitud del inciso 1.2 (optimizando sobre logaritmos, con Nelder–Mead desde dos puntos de partida). El experimento se repitió con 30 semillas (0 a 29). Criterio de éxito: que la media de los estimados quede cerca del valor verdadero, con una dispersión razonable.

| | $r$ | $\alpha$ | $a$ | $b$ |
|---|---|---|---|---|
| Verdadero | 0.25 | 4.0 | 0.8 | 2.5 |
| Media (30 semillas) | 0.250 | 4.017 | 0.851 | 2.719 |
| Desv. est. | 0.014 | 0.293 | 0.148 | 0.659 |
| Desv. est. / media | 5.5% | 7.3% | 17.4% | 24.2% |

**¿Recuperamos los parámetros?** Sí. $r$ y $\alpha$ se recuperan casi exactamente. $a$ y $b$ quedan por encima del valor verdadero (+6% y +9%). Con 30 réplicas el error estándar de la media es $sd/\sqrt{30}$, es decir 0.027 para $a$ y 0.120 para $b$, y los sesgos (0.051 y 0.219) equivalen a ≈1.9 errores estándar: es una diferencia pequeña frente a la dispersión de *una* estimación, pero no frente a la incertidumbre de la *media*, así que no la descartamos como ruido. Es el sesgo de muestra finita típico del máximo verosímil en parámetros poco identificados. Con 2,357 clientes es moderado, y el código recupera los parámetros que se le imponen, por lo que es razonable aplicarlo a datos reales sabiendo que $a$ y $b$ pueden salir algo altos.

**¿Cuáles se estiman con más precisión y por qué?** $r$ y $\alpha$, los parámetros de la tasa de compra, son los más precisos (error relativo de ~5% a 7%). $a$ y $b$, los de la probabilidad de morir, son los menos precisos, sobre todo $b$ (~24%).

La razón es de dónde sale la información. Sobre $\lambda$ informan todas las compras observadas de todos los clientes. Sobre $p$, en cambio, la única señal es cuánto tiempo pasó desde la última compra, porque la muerte nunca se observa directamente: se infiere. Un cliente que lleva mucho tiempo sin comprar pudo haber muerto o simplemente ser un comprador lento, y los dos efectos se confunden. Además, $a$ y $b$ describen la forma de la distribución de $p$ y no solo su media, lo que exige aún más información de la que aportan los datos.

# 2.1 Ajuste a CDNOW

Se estimaron $(r,\alpha,a,b)$ maximizando la log-verosimilitud con los datos de calibración (39 semanas). Se optimizó sobre los logaritmos de los parámetros, con Nelder–Mead, desde tres puntos de partida distintos. Los tres convergieron al mismo resultado.

| Parámetro | Estimado (este trabajo) | Fader, Hardie y Lee (2005) |
|---|---|---|
| $r$ | 0.243 | 0.243 |
| $\alpha$ | 4.414 | 4.414 |
| $a$ | 0.793 | 0.793 |
| $b$ | 2.426 | 2.426 |
| Log-verosimilitud | −9582.4 | −9582.4 |

Los estimados coinciden con los del artículo, lo que valida la preparación de los datos $(x, t_x, T)$ y la implementación de la verosimilitud.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gamma, beta

print("Tasa de compra promedio (por semana):", round(r_hat / alpha_hat, 4))
print("Prob. media de morir tras una compra:", round(a_hat / (a_hat + b_hat), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

lam = np.linspace(0.001, 0.5, 400)
ax[0].plot(lam, gamma.pdf(lam, a=r_hat, scale=1 / alpha_hat))   # scale = 1/tasa
ax[0].set_ylim(0, 15)
ax[0].set_title("Heterogeneidad en la tasa de compra λ")
ax[0].set_xlabel("λ (compras por semana)")

p = np.linspace(0.001, 0.999, 400)
ax[1].plot(p, beta.pdf(p, a_hat, b_hat))
ax[1].set_ylim(0, 8)
ax[1].set_title("Heterogeneidad en la prob. de morir p")
ax[1].set_xlabel("p")

plt.tight_layout()
plt.show()

# 2.2 Interpretación de los parámetros

- **Tasa de compra promedio de un cliente activo:** $E[\lambda]=r/\alpha = 0.243/4.414 \approx 0.055$ compras por semana. Ojo: eso *no* equivale a "una compra cada ~18 semanas". Como $r<1$, el tiempo medio entre compras $E[1/\lambda]$ es infinito (la densidad de $\lambda$ se dispara en 0 y $1/\lambda$ no tiene media), así que $1/E[\lambda]$ no es el tiempo típico entre compras; la mediana de $\lambda$ (abajo) describe mejor a un cliente típico.
- **Probabilidad promedio de morir después de una compra:** $a/(a+b) = 0.793/3.219 \approx 0.246$ (24.6%).

**Qué dicen las densidades sobre qué tan parecidos son los clientes:**

Los clientes son muy distintos entre sí. Como $r<1$, la densidad de $\lambda$ es decreciente y se dispara cerca de 0: aproximadamente la mitad de los clientes tiene $\lambda<0.01$ (menos de una compra cada 100 semanas) y cerca del 73% tiene $\lambda<0.05$. En el otro extremo, solo ~6% compra más de una vez cada 4 semanas ($\lambda>0.25$). Por eso la media (0.055) no representa al cliente típico: la mayoría está muy por debajo y una minoría muy activa la sube.

Con la probabilidad de morir pasa algo parecido. Como $a<1$, la densidad de $p$ también se concentra cerca de 0: ~32% de los clientes tiene $p<0.1$, y solo ~14% tiene $p>0.5$. Hay muchos clientes "leales" que casi no abandonan y una minoría con alta probabilidad de irse tras cada compra.

En resumen, no existe un "cliente promedio": la base mezcla una mayoría de compradores muy esporádicos con una minoría activa, y eso es lo que el modelo captura con las distribuciones gamma y beta.

In [ ]:
B = 200
rng_boot = np.random.default_rng(123)
n = len(x)
inicio_boot = np.log([r_hat, alpha_hat, a_hat, b_hat])    # partir de los estimados reales

boot = []
for _ in range(B):
    idx = rng_boot.integers(0, n, size=n)             # remuestreo con reemplazo: n clientes
    o = minimize(lambda lp: -log_verosimilitud(np.exp(lp), x[idx], t_x[idx], T[idx]),
                 inicio_boot, method="Nelder-Mead",
                 options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8})
    boot.append(np.exp(o.x))

boot = np.array(boot)
print("Estimados:  ", np.array([r_hat, alpha_hat, a_hat, b_hat]).round(3))
print("Error est.: ", boot.std(axis=0, ddof=1).round(3))
print("IC 95%:")
print(np.percentile(boot, [2.5, 97.5], axis=0).round(3))   # IC 95% por percentiles

# 2.3 Error estándar con bootstrap (bono)

**Método.** Se remuestrearon con reemplazo los 2,357 clientes (cada cliente aporta su terna $(x, t_x, T)$ completa), y en cada remuestreo se reestimaron $(r,\alpha,a,b)$ con la misma verosimilitud, partiendo de los estimados originales. Se hicieron 200 remuestreos. El error estándar es la desviación estándar de los estimados bootstrap, y el intervalo del 95% usa los percentiles 2.5 y 97.5.

Se remuestrean clientes y no transacciones sueltas, porque la unidad de observación del modelo es el cliente: $(x, t_x, T)$ resume toda su historia. Remuestrear filas sueltas rompería esas historias.

| | $r$ | $\alpha$ | $a$ | $b$ |
|---|---|---|---|---|
| Estimado | 0.243 | 4.414 | 0.793 | 2.426 |
| Error estándar (bootstrap) | 0.014 | 0.469 | 0.173 | 0.687 |
| IC 95% | [0.214, 0.270] | [3.553, 5.359] | [0.545, 1.207] | [1.530, 3.965] |
| Desv. est. en la simulación (1.3) | 0.014 | 0.293 | 0.148 | 0.659 |

**Comparación con la simulación.** Ambas fuentes cuentan la misma historia: $r$ es el parámetro mejor identificado y $b$ el peor, con $a$ y $\alpha$ en medio. Los errores de $r$, $a$ y $b$ son muy parecidos entre el bootstrap y la simulación. El de $\alpha$ sale más alto en el bootstrap (0.47 frente a 0.29). Una posible explicación es que la simulación usa datos que cumplen exactamente el modelo y con parámetros distintos ($\alpha=4.0$), mientras que el bootstrap parte de datos reales, donde el modelo es solo una aproximación.

**Lectura práctica.** El intervalo de $b$ es muy ancho ([1.53, 3.97]): los datos no permiten precisar bien cuánto abandonan los clientes después de una compra, aunque sí dicen que $r$ y la media de la tasa de compra están bien determinados.